# Adquisición, adaptación y selección de datos abiertos de tráfico para el gemelo digital de movilidad urbana

Este notebook implementa una parte del sistema de adquisición y preprocesamiento de datos del Trabajo Fin de Máster, centrada en la obtención y preparación de datos abiertos de tráfico de la ciudad de Madrid.

A partir de la información pública emitida por el Ayuntamiento de Madrid, se lleva a cabo un proceso compuesto por las siguientes etapas:

* Descarga de datos en formato XML
* Limpieza y adaptación de campos
* Transformación de coordenadas
* Filtrado espacial de la zona de estudio
* Exportación del resultado a un formato reutilizable

## Objetivo del módulo

Este notebook forma parte del módulo de adquisición, adaptación y procesamiento de datos abiertos del sistema propuesto en el TFM.

Su finalidad es obtener información real procedente de sensores de tráfico y transformarla en un conjunto de datos preparado para su integración en fases posteriores del gemelo digital.

En este trabajo se toma como caso de estudio una zona concreta de Madrid, correspondiente al entorno de Argüelles, con el fin de trabajar sobre un escenario acotado, manejable y suficientemente representativo.

In [ ]:
# Instalación de dependencias necesarias en Google Colab
!pip install pyproj shapely folium -q
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from pyproj import Transformer
from shapely.geometry import Point, Polygon
import folium

## Definición de la fuente de datos y la zona de estudio

En esta sección se presenta la fuente de datos empleada, los nombres de los ficheros de trabajo y el polígono aproximado utilizado para delimitar la zona de estudio.

In [ ]:
# URL de datos abiertos del Ayuntamiento de Madrid
URL_DATOS = "https://informo.madrid.es/informo/tmadrid/pm.xml"

# Nombre de archivo temporal
XML_LOCAL = "pm.xml"

# Nombre de salida del dataset procesado
CSV_SALIDA = "sensores_arguelles.csv"

# Polígono aproximado de la zona de estudio (Argüelles)
# Coordenadas en formato (longitud, latitud)
zona_estudio_coords = [
    (-3.725290, 40.432520),
    (-3.719366, 40.435115),
    (-3.716179, 40.430848),
    (-3.713025, 40.426596),
    (-3.710715, 40.423434), 
    (-3.713325, 40.421769),
    (-3.716485, 40.424959),
    (-3.718561, 40.425441),
]

## Descarga del fichero XML

En esta fase se descarga el fichero XML que contiene la información de los sensores de tráfico publicada por el Ayuntamiento de Madrid y se almacena localmente para su posterior procesamiento.

In [ ]:
response = requests.get(URL_DATOS, timeout=30)
response.raise_for_status()

with open(XML_LOCAL, "wb") as f:
    f.write(response.content)

print("Descarga completada correctamente.")
print(f"Tamaño del fichero descargado: {len(response.content)} bytes")

Descarga completada correctamente.
Tamaño del fichero descargado: 2183626 bytes


## Conversión del XML a DataFrame

Una vez descargado el fichero, se parsea su contenido y se construye una estructura tabular en formato DataFrame de pandas, en la que cada fila representa un sensor y cada columna uno de sus atributos.

In [ ]:
tree = ET.parse(XML_LOCAL)
root = tree.getroot()

records = []
for pm in root.findall(".//pm"):
    record = {}
    for child in pm:
        record[child.tag] = child.text
    records.append(record)

df = pd.DataFrame(records)

print("Número total de sensores obtenidos:", df.shape[0])
print("Número de columnas:", df.shape[1])

df.head()

Número total de sensores obtenidos: 4881
Número de columnas: 13


,idelem,descripcion,accesoAsociado,intensidad,ocupacion,carga,nivelServicio,intensidadSat,error,subarea,st_x,st_y,velocidad
0,9841,Valle de Mena S-E - Acc.Ramon Castroviejo-Gta....,0301005,999,7,28,1,3100,N,0328,"438339,375874991","4480454,96970565",NaN
1,9843,Dr.Ramon Castroviejo O-E - Av.Miraflores-San M...,0301003,960,38,42,1,2000,N,0328,"438098,880458143","4480455,13738494",NaN
2,9842,Nueva Zelanda S-N - Isla Cristina-Gta.Isaac Rabín,0301001,160,3,22,0,700,N,0328,"438305,950926946","4480365,97449961",NaN
3,11396,Dr. Ramon Castroviejo E-O - Islas Bikini-Gta I...,0301004,880,2,22,0,3150,N,0328,"438345,801923105","4480548,97911408",NaN
4,11243,Islas Aleutianas N-S - Manuel Garrido -Isaac R...,0301002,240,1,15,0,1350,N,0328,"438226,45123268","4480578,31957825",NaN


## Inspección inicial de los datos

Antes de continuar con el preprocesamiento, se revisan las columnas disponibles y algunos campos de interés con el objetivo de comprobar la estructura del conjunto de datos y verificar que contiene la información necesaria para las etapas posteriores.

In [ ]:
print("Columnas disponibles:")
print(list(df.columns))

Columnas disponibles:
['idelem', 'descripcion', 'accesoAsociado', 'intensidad', 'ocupacion', 'carga', 'nivelServicio', 'intensidadSat', 'error', 'subarea', 'st_x', 'st_y', 'velocidad']


In [ ]:
columnas_interes = [c for c in ["idelem", "descripcion", "intensidad", "ocupacion", "carga", "st_x", "st_y"] if c in df.columns]
df[columnas_interes].head(10)

,idelem,descripcion,intensidad,ocupacion,carga,st_x,st_y
0,9841,Valle de Mena S-E - Acc.Ramon Castroviejo-Gta....,999,7,28,"438339,375874991","4480454,96970565"
1,9843,Dr.Ramon Castroviejo O-E - Av.Miraflores-San M...,960,38,42,"438098,880458143","4480455,13738494"
2,9842,Nueva Zelanda S-N - Isla Cristina-Gta.Isaac Rabín,160,3,22,"438305,950926946","4480365,97449961"
3,11396,Dr. Ramon Castroviejo E-O - Islas Bikini-Gta I...,880,2,22,"438345,801923105","4480548,97911408"
4,11243,Islas Aleutianas N-S - Manuel Garrido -Isaac R...,240,1,15,"438226,45123268","4480578,31957825"
5,5422,Av.Betanzos S-N - Av.Monforte de Lemos-Av.Ilus...,300,1,33,"439614,078998791","4481136,52322837"
6,5423,Av.Ilustración O-E - M-30 -Av.Betanzos,1560,18,36,"439457,354152262","4481247,34531919"
7,5424,(AFOROS)Av.Ilustración E-O - Centro Cívico-Av....,2860,11,43,"439730,39599613","4481359,58768476"
8,5425,(AFOROS)Av.Ilustración O-E - Av.Betanzos-Centr...,2100,11,49,"439731,14693176","4481347,95261854"
9,5426,Av.Ilustración O-E - Centro Cívico-Ginzo de Limia,3140,13,59,"440086,941698542","4481441,92142802"


## Limpieza y adaptación de coordenadas

Las coordenadas originales se encuentran almacenadas como texto y expresadas en un sistema de referencia proyectado. En esta etapa se convierten a formato numérico y se preparan para su posterior transformación a latitud y longitud.

In [ ]:
df["st_x_num"] = pd.to_numeric(df["st_x"].str.replace(",", ".", regex=False), errors="coerce")
df["st_y_num"] = pd.to_numeric(df["st_y"].str.replace(",", ".", regex=False), errors="coerce")

total_inicial = df.shape[0]

df = df.dropna(subset=["st_x_num", "st_y_num"]).copy()

total_validas = df.shape[0]
total_descartadas = total_inicial - total_validas

print("Sensores iniciales:", total_inicial)
print("Sensores con coordenadas válidas:", total_validas)
print("Sensores descartados por coordenadas no válidas:", total_descartadas)

Sensores iniciales: 4881
Sensores con coordenadas válidas: 4881
Sensores descartados por coordenadas no válidas: 0


## Transformación de coordenadas a latitud y longitud
Las coordenadas originales se encuentran en el sistema ETRS89 / UTM zona 30N (EPSG:25830), un sistema de coordenadas proyectadas que expresa las posiciones en metros y que es habitual en datos geográficos a nivel local en Europa.

En esta etapa se transforman al sistema WGS84 (EPSG:4326), basado en latitud y longitud, que constituye el estándar utilizado en la mayoría de herramientas de visualización geográfica y servicios de mapas basados en GPS. Esta transformación permite representar los datos en mapas y facilita su integración con otros sistemas dentro del gemelo digital de movilidad urbana.

In [ ]:
transformer = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(df["st_x_num"].values, df["st_y_num"].values)

df["lon"] = lon
df["lat"] = lat
df[["idelem", "descripcion", "st_x_num", "st_y_num", "lat", "lon"]].head()

,idelem,descripcion,st_x_num,st_y_num,lat,lon
0,9841,Valle de Mena S-E - Acc.Ramon Castroviejo-Gta....,438339.375875,4.480455e+06,40.472488,-3.727397
1,9843,Dr.Ramon Castroviejo O-E - Av.Miraflores-San M...,438098.880458,4.480455e+06,40.472472,-3.730234
2,9842,Nueva Zelanda S-N - Isla Cristina-Gta.Isaac Rabín,438305.950927,4.480366e+06,40.471684,-3.727783
3,11396,Dr. Ramon Castroviejo E-O - Islas Bikini-Gta I...,438345.801923,4.480549e+06,40.473335,-3.727331
4,11243,Islas Aleutianas N-S - Manuel Garrido -Isaac R...,438226.451233,4.480578e+06,40.473591,-3.728741


## Definición del polígono de la zona de estudio y filtrado espacial

Una vez transformadas las coordenadas, se utiliza el polígono que delimita la zona de estudio correspondiente al entorno de Argüelles anteriormente definido y se seleccionan únicamente los sensores situados en su interior.

In [ ]:
poligono_arguelles = Polygon(zona_estudio_coords)
df_arguelles = df[
    df.apply(lambda r: poligono_arguelles.contains(Point(r["lon"], r["lat"])), axis=1)
].copy()

print("Número de sensores dentro de la zona de estudio:", df_arguelles.shape[0])

Número de sensores dentro de la zona de estudio: 51


In [ ]:
columnas_mostrar = [c for c in ["idelem", "descripcion", "intensidad", "ocupacion", "carga", "lat", "lon"] if c in df_arguelles.columns]
df_arguelles[columnas_mostrar].head(20)

,idelem,descripcion,intensidad,ocupacion,carga,lat,lon
2266,4284,PL. ESPANA N-S(SAN LEONARDO-CUESTA SAN VICENTE),396,5,23,40.424096,-3.711602
2267,4285,CUESTA SAN VICENTE O-E(GRAN VIA-FERRAZ),144,3,19,40.423108,-3.711576
2268,4286,PL. ESPANA S-N(REYES-SAN LEONARDO),1440,9,50,40.423728,-3.711180
2283,4353,(AFOROS) PRINCESA S-N (SAN LEONARDO-VENTURA RO...,1480,13,56,40.425439,-3.712459
2296,10874,FERRAZ N-S (IRUN -CUESTA SAN VICENTE)(GIRO),1368,10,46,40.422521,-3.713901
2297,4313,PL.ESPANA O-E(BAILEN-GRAN VIA),936,12,37,40.422237,-3.712938
2300,4316,FERRAZ N-S(IRUN- CUESTA SAN VICENTE)(FRENTE),612,12,44,40.422550,-3.713854
2306,10885,(TACTICA) PL.ESPANA E-O( GRAN VIA -FERRAZ),72,85,86,40.422282,-3.713076
2307,10875,FERRAZ N-S (P.M 16045+16041 ),1116,5,26,40.422402,-3.713734
2309,4338,PRINCESA S-N(PL. MONCLOA-Po MORET),920,9,33,40.434225,-3.718790


## Visualización de los sensores seleccionados

A continuación, se representa la zona de estudio junto con la localización de los sensores filtrados, con el fin de comprobar visualmente la coherencia del resultado obtenido.

In [ ]:
mapa = folium.Map(location=[40.4285, -3.7170], zoom_start=15)

folium.Polygon(
    locations=[(lat, lon) for lon, lat in zona_estudio_coords],
    color="blue",
    weight=3,
    fill=False,
    tooltip="Zona de estudio"
).add_to(mapa)

for _, r in df_arguelles.iterrows():
    texto_popup = f"""
    <b>Sensor:</b> {r.get('idelem', 'N/D')}<br>
    <b>Descripción:</b> {r.get('descripcion', 'N/D')}<br>
    <b>Intensidad:</b> {r.get('intensidad', 'N/D')}
    """
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=4,
        color="red",
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(texto_popup, max_width=300)
    ).add_to(mapa)
mapa

## Resumen del preprocesamiento realizado

En esta sección se presentan algunos indicadores básicos que permiten sintetizar el resultado del proceso de adquisición, limpieza, transformación y filtrado espacial aplicado a los datos.

In [ ]:
resumen = pd.DataFrame({
    "Métrica": [
        "Sensores descargados",
        "Sensores con coordenadas válidas",
        "Sensores en la zona de estudio"
    ],
    "Valor": [
        total_inicial,
        total_validas,
        df_arguelles.shape[0]
    ]
})
resumen

,Métrica,Valor
0,Sensores descargados,4881
1,Sensores con coordenadas válidas,4881
2,Sensores en la zona de estudio,51


## Exportación del conjunto de datos procesado

Finalmente, el conjunto de sensores pertenecientes a la zona de estudio se exporta a un fichero CSV. Este archivo constituye la salida estructurada del módulo de preprocesado y puede emplearse como entrada en etapas posteriores del gemelo digital.

In [ ]:
df_arguelles.to_csv(CSV_SALIDA, index=False, encoding="utf-8")

print(f"Fichero exportado correctamente: {CSV_SALIDA}")
print(f"Número de registros exportados: {df_arguelles.shape[0]}")

Fichero exportado correctamente: sensores_arguelles.csv
Número de registros exportados: 51


In [ ]:
from google.colab import files
files.download(CSV_SALIDA)

ModuleNotFoundError: No module named 'google.colab'

## Conclusiones

El proceso implementado permite obtener y transformar datos abiertos de tráfico en un formato estructurado, georreferenciado y reutilizable. El resultado constituye una entrada válida para fases posteriores del gemelo digital de movilidad urbana, especialmente aquellas relacionadas con la representación del estado del tráfico y la generación de escenarios de análisis.

Además, el enfoque seguido permite aislar esta funcionalidad como un **módulo independiente** dentro de la arquitectura general del sistema, lo que favorece su reutilización, mantenimiento y posible ampliación futura.